# Weather × Commerce: Evidence-First Exploratory Data Analysis

Four **separate case studies** use a common EDA workflow. Their locations, cadences and value definitions differ, so coefficients are not ranked as though they came from one experiment. Historical EDA uses observed weather only. Estimated revenue, proxy weather and scenarios are labelled and excluded from unsupported causal claims.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
from src.real_eda_engine import (build_dataset_scorecards, get_matched_series_data,
    compute_quality_evidence, compute_missingness_matrix, compute_eda_statistics,
    compute_correlation_and_math, compute_advanced_association)
scorecards = pd.DataFrame(build_dataset_scorecards())
display(scorecards)
assert scorecards['key'].nunique() == 4


In [ ]:
series_key = 'nyc_taxi'
d = get_matched_series_data(series_key)
display(d.head())
print({'rows':len(d), 'start':d.date.min(), 'end':d.date.max()})
assert not d['weather_estimated'].any()


In [ ]:
quality = compute_quality_evidence(d)
quality_table = pd.DataFrame(quality['rows'])
display(quality_table[['feature','dtype','n','missing','missing_pct','missing_mechanism']])
print({'duplicate_rows':quality['duplicate_rows'], 'duplicate_dates':quality['duplicate_dates']})


In [ ]:
display(quality_table[['feature','invalid_negative','iqr_flags','isolation_flags']])
assert (quality_table['iqr_flags'] >= 0).all()


In [ ]:
display(quality_table[['feature','raw_skew','log1p_skew']])


In [ ]:
stats = pd.DataFrame(compute_eda_statistics(d))
display(stats)


In [ ]:
corr, columns, association = compute_correlation_and_math(d)
display(corr)
display(pd.DataFrame(association['associations']))


In [ ]:
worked = association['rain']
result = worked
display(pd.Series({k:worked[k] for k in ['n','xmean','ymean','sxx','syy','sxy','covariance','pearson','spearman','slope','intercept','r2']}).to_frame('value'))
paired = d[['rain_mm','count']].dropna()
assert np.isclose(worked['pearson'], paired.rain_mm.corr(paired['count']))


In [ ]:
ordered = d.sort_values('date').copy()
ordered['gap_days'] = ordered.date.diff().dt.total_seconds().div(86400)
print(ordered.nlargest(5, 'gap_days')[['date','gap_days']].to_string(index=False))
ax = ordered.set_index('date')['count'].plot(figsize=(12,3), style='o', ms=2, linestyle='None')
ax.set_title('Observed activity points; no line is drawn across missing periods')
display(ax.figure)
plt.close(ax.figure)


In [ ]:
diagnostics = compute_advanced_association(d)
display(pd.Series(diagnostics).drop(labels=['vif','sensitivity','lagged'], errors='ignore').to_frame('value'))
if diagnostics['available']:
    display(pd.DataFrame(diagnostics['vif']))


In [ ]:
if diagnostics['available']:
    display(pd.DataFrame(diagnostics['sensitivity']))
    display(pd.DataFrame(diagnostics['lagged']))
else:
    print(diagnostics['reason'])


In [ ]:
conclusions = {
 'observed': 'Descriptive distributions, time patterns and matched-weather associations.',
 'estimated': 'Citi Bike fare revenue and executive scenario outputs; excluded from unsupported causal claims.',
 'not_identified': 'Individual behaviour changes or universal weather effects across cases.'}
display(pd.Series(conclusions).to_frame('claim boundary'))
